#### **Step 2 - Sparse Retrieval (BM25)** 

- Sparse Retrieval BM25 is a form of keyword search that checks for exact keyword matches. BM25 operates purely on whitespace-split words from the raw text 

- For a query and a chunk, **BM25 produces a relevance score** by summing a per-term score over every query term that appears in the chunk (keyword matching)

- **2 Main Components of BM25**
    1. **Term Frequency (TF)**
        - If the query word "championship" appears in a chunk, that's a signal of relevance.

        - If it appears 5 times, that's still a signal - but not 5 times as strong a signal 

        - BM25 applies a saturating function to term frequency, so going from 1 occurence to 2 matters a lot, but going from 10 to 11 barely matters.

        - This prevents a chunk that just happens to repeat a word many times from dominating purely on repetition 

        **TF Formula:**

        $\text{TF}(term, chunk) = \frac{tf \times (k_1 + 1)}{tf + k_1}$

        - $tf$ = raw count of how many times the term appears in the chunk
        - $k_1$ (≈1.2–2.0) = controls how quickly the score saturates

        **Behavior as $tf$ increases** (with $k_1 = 1.5$):

        | $tf$ | 0 | 1 | 2 | 3 | 5 | 10 | 20 |
        |---|---|---|---|---|---|---|---|
        | score | 0 | 1.0 | 1.43 | 1.67 | 2.0 | 2.35 | 2.65 |

        **Problem: Longer chunks have a greater chance of containing more instances of a key word creating a bias towards longer chunks**

        **Length Normalization**
        - BM25 corrects for this by comparing each chunk's length to the corpus average, and adjusting the TF score up or down accordingly

        - This ensures long chunks do not rank higher simply because it has more chances to contain the query word

        **Full TF Formula**

        $\text{TF}(term, chunk) = \frac{tf \times (k_1 + 1)}{tf + k_1 \times \left(1 - b + b \times \dfrac{|chunk|}{avgChunkLen}\right)}$

        - $b$ (≈0.75) = controls strength of length normalization (ranges from 0 to 1 - no length normalisation to full length normalisation)
            - The `1-b+b×ratio` formula is built so b acts as a safe dial — at b=0 it neutralizes to 1 (no length penalty), and at b=1 it fully applies the length ratio, with everything in between smoothly blending the two.
        - $|chunk|$ = number of words in this chunk
        - $avgChunkLen$ = average chunk length (word count) across the corpus

        **How the ratio $|chunk| / avgChunkLen$ drives the correction:**
        - **Longer than average** → ratio $> 1$ → denominator grows → TF score is *penalized* (pulled down)
        - **Shorter than average** → ratio $< 1$ → denominator shrinks → TF score gets a relative *boost*

    #
    2. **Inverse Document Frequency (IDF)** 
        - Common words like "the", "is", "player" appear in almost every chunk and therefore carry very little information about which chunk is actually relevant.

        - Rare words like "Faker" or "Kpop" appearing in a chunk are much stronger signals

        - IDF downweights common terms and upweights rare ones - this is exactly why BM25 style search doesn't just degenerate into "the chunk witht he most word matches wins"

        - Downweighting by IDF works by multiplying the term frequency by an appropriate value to either upweight or downweight the word's relevance based on rarity

        **IDF Formula:** 

        $\text{IDF}(term) = \log\left(\frac{N - n + 0.5}{n + 0.5}\right)$         
        - $N$ = total number of chunks in the corpus        
        - $n$ = number of chunks that contain this term


        **Intuition for the formula:**
        - If a term appears in almost every chunk ($n \approx N$), the ratio $(N-n+0.5)/(n+0.5)$ approaches $0.5/(n+0.5)$, a small number — $\log$ of a small number is small (or even negative), so common words score near zero or get suppressed entirely

        - If a term appears in very few chunks ($n$ small relative to $N$), the ratio is large — $\log$ of a large number gives a high IDF score, so rare words are weighted heavily

        - The $+0.5$ in both numerator and denominator is a smoothing constant that prevents division by zero and stabilizes the score when $n$ is very small or very close to $N$
    
    #
    #

    **Combining each of the 3 components - For a given chunk and query**

    $\text{BM25}(chunk, query) = \sum\limits_{term \in query} \text{IDF}(term) \times \frac{tf(term, chunk) \times (k_1 + 1)}{tf(term, chunk) + k_1 \times \left(1 - b + b \times \dfrac{|chunk|}{avgChunkLen}\right)}$


    1. Split the query into individual words
    2. For each word, compute its BM25 contribution: `IDF(word) × TF(word, chunk)`
    3. Sum all these per-word contributions together to achieve a score for the chunk (sum of scores for all terms in the query evaluated against the chunk)
    4. Perform these operations on each of the chunks and rank them based on their score

In [1]:
from rank_bm25 import BM25Okapi

class SparseIndex:
    def __init__(self, chunks):
        # we want to use these chunks in different methods so intialise it 
        self.chunks = chunks 

        # BM25 computed on individual words so we want to tokenise strings into a list of words
        tokenised_chunks = [chunk['text'].lower().split() for chunk in chunks]

        # intialise the object to calculate BM25 
        # pass through tokenised_chunks - similar to .fit() a model
        # we pass through the tokenised_chunks so it can compute chunk metrics used in IDF and TF 
        # - computes avgChunkLen (average word count across all chunks)
        # - computes document frequency (n) for every unique word in the corpus - how many chunks each word appears in 
        self.bm25 = BM25Okapi(tokenised_chunks) # essentially precomputes generic chunk metrics which will be reused across searches with different queries 

    def search(self, query, k=5):
        # just like how we tokenised chunks we want to tokenise the query into a list of words as well
        tokenised_query = query.lower().split()

        # use our constructed object with the chunks passed to compute a bm255 score for each chunk against the query 
        # return an array of floats for each chunk (in same indexed order as the chunks we passed through)
        scores = self.bm25.get_scores(tokenised_query) 

        # we want to pair each dict in chunk with its corresponding score to create a list of (chunk_dict, score) tuples
        # this allows chunks to be paired with their corresponding scores depending on the query while keeping the 2 as individual objects
        paired = zip(self.chunks, scores) # creates a zip object which will pair the 2 into tuples and when sortedd the zip object becomes a lisat

        # we then want to rank these (chunk_dict, score) tuples by score
        ranked = sorted(paired, key=lambda pair: pair[1], reverse=True) 

        # return the top k chunks (chunk_dict, score)
        return ranked[:k]

**`SparseIndex` Intuitive Summary**

**The Big Picture:** For every chunk in the corpus (collection of chunks), sum the BM25 contribution of each word in the query against that chunk, producing one score per chunk. Repeating this across all chunks gives a full list of scores, which are then ranked to find the most relevant chunks

$\text{BM25}(chunk, query) = \sum\limits_{term \in query} \text{IDF}(term) \times TF(term,chunk)$


- **Build time (`__init__`)** 
    - We want to first tokenise every chunk into lowercase words
    - Then pass that corpus into `BM25Okapi`, which precomputes the corpus-wide statistics (document frequency per word, average chunk length) needed for IDF and TF - this is the expensive, build-once step

- **Query time (`.search()`)** — only the query-specific work happens here for each individual chunk:
    - The query is tokenized the same way as the corpus (lowercase + split)
    - For each query word, its precomputed IDF and per-chunk $tf$ are looked up (not recomputed) and multiplied together per chunk
    - These per-word contributions are summed across all query words to get each chunk's final score

- **Ranking** — once every chunk has a score, `sorted()` orders them highest-first, and the top-k are returned as `(chunk_dict, score)` tuples.
#
- **Why split it this way** — the expensive corpus-wide statistics (IDF, avgChunkLen) only need to be computed once, at build time. Every subsequent `.search()` call reuses them instead of recalculating from scratch, making repeated queries fast.

**Testing**

In [2]:
from chunking import load_documents, chunk_documents 

docs = load_documents("../data")
chunks = chunk_documents(docs)
sparse = SparseIndex(chunks)

results = sparse.search("How many championships has Faker won?", k=3)
for chunk, score in results:
    print(f"score={score:.3f} | {chunk['text'][:100]}...")

score=4.736 |  player in the history of the game.

Outside of Worlds, Faker has won the Mid-Season Invitational tw...
score=3.106 |  constantly, which is one reason longevity is considered such an achievement.

Championships in trad...
score=2.053 |  reinventing his role. He shifted from a pure carry into a strategic leader who set the tempo of the...
